In [1]:
!pip -q install -U transformers datasets trl peft accelerate bitsandbytes wandb huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 58.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 68.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 94.2 MB/s eta 0:00:00:00:01


In [2]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
WANDB_API_KEY = secrets.get_secret("WANDB_API_KEY")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["WANDB_API_KEY"] = WANDB_API_KEY

In [3]:
from huggingface_hub import login
import wandb

login(token=HF_TOKEN)
wandb.login(key=WANDB_API_KEY)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: leminhhuy-008071 (leminhhuy-008071-ho-chi-minh-city-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DATASET_REPO = "huyle1901/legal-rag-splits"   # dataset bạn đã upload
OUTPUT_DIR = "/kaggle/working/qwen25_3b_qlora_legal"

MAX_SEQ_LEN = 1536
USE_QLORA = True

# train config
EPOCHS = 2
LR = 2e-4
BATCH_SIZE = 1
GRAD_ACC = 8
LOG_STEPS = 10
EVAL_STEPS = 100
SAVE_STEPS = 100

In [5]:
from huggingface_hub import snapshot_download
from datasets import load_dataset

local_ds = snapshot_download(repo_id=DATASET_REPO, repo_type="dataset", token=HF_TOKEN)

train_path = f"{local_ds}/training_lora/train_messages.jsonl"
val_path = f"{local_ds}/training_lora/val_messages.jsonl"

raw_ds = load_dataset(
    "json",
    data_files={"train": train_path, "validation": val_path})

raw_ds

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'messages', 'meta'],
        num_rows: 2445
    })
    validation: Dataset({
        features: ['id', 'messages', 'meta'],
        num_rows: 150
    })
})

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_QLORA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

model.config.use_cache = False


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [7]:
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

train_cols = raw_ds["train"].column_names
ds = raw_ds.map(format_chat, remove_columns=train_cols)

print(ds["train"][0])
print(ds["train"][0]["text"][:1000])


Map:   0%|          | 0/2445 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

{'text': '<|im_start|>system\nBạn là trợ lý pháp lý Việt Nam. Chỉ trả lời dựa trên thông tin được cung cấp trong bộ dữ liệu huấn luyện.<|im_end|>\n<|im_start|>user\nCau hoi:\nSoft OTP là hình thức xác thực thế nào, gồm những loại nào và phần mềm Soft OTP cần tuân thủ những yêu cầu cụ thể gì theo quy định?<|im_end|>\n<|im_start|>assistant\nCăn cứ tại điểm d khoản 3 Điều 11 Thông tư 50/2024/TT-NHNN quy định về Soft OTP là hình thức xác nhận thông qua mã OTP được tạo bởi phần mềm cài đặt trên thiết bị di động của khách hàng, phần mềm Soft OTP có thể là phần mềm độc lập hoặc được tích hợp với phần mềm ứng dụng Mobile Banking.\n- Soft OTP có 02 loại:\n+ Soft OTP loại cơ bản: Mã OTP được sinh ngẫu nhiên theo thời gian, đồng bộ với hệ thống Online Banking;\n+ Soft OTP loại nâng cao: Mã OTP được tạo kết hợp với mã của từng giao dịch, khi thực hiện giao dịch, hệ thống Online Banking tạo ra một mã giao dịch thông báo cho khách hàng hoặc truyền cho phần mềm Soft OTP, khách hàng hoặc phần mềm Soft

In [13]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

In [14]:
import torch
import trl, transformers
from trl import SFTConfig, SFTTrainer

print("trl:", trl.__version__, "transformers:", transformers.__version__)

train_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    logging_steps=LOG_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    report_to="wandb",
    run_name="qwen25-3b-qlora-legal-sft",
    max_length=MAX_SEQ_LEN,        # thay max_seq_length
    dataset_text_field="text",     # chuyển vào SFTConfig
    packing=False,
    warmup_steps=100,              # thay warmup_ratio
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
)

trainer = SFTTrainer(
    model=model,
    args=train_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    peft_config=peft_config,
    processing_class=tokenizer,  
)


trl: 1.1.0 transformers: 5.5.3


Adding EOS to train dataset:   0%|          | 0/2445 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2445 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

In [15]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
100,0.930835,0.924570
200,0.810380,0.826398
300,0.737477,0.763999
400,0.637839,0.726050
500,0.625772,0.699730
600,0.666361,0.688586
612,0.625056,0.688512


TrainOutput(global_step=612, training_loss=0.7920817725019518, metrics={'train_runtime': 33179.7308, 'train_samples_per_second': 0.147, 'train_steps_per_second': 0.018, 'total_flos': 4.521516924434842e+16, 'train_loss': 0.7920817725019518})

In [17]:
from huggingface_hub import HfApi, upload_folder
from pathlib import Path

ADAPTER_REPO = "huyle1901/qwen25-3b-legal-qlora"
OUT = Path(OUTPUT_DIR)

# save local
trainer.model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)

# create repo if needed
api = HfApi(token=HF_TOKEN)
api.create_repo(ADAPTER_REPO, repo_type="model", private=False, exist_ok=True, token=HF_TOKEN)

# upload whole output dir (adapter + tokenizer files)
upload_folder(
    repo_id=ADAPTER_REPO,
    repo_type="model",
    folder_path=str(OUT),
    path_in_repo="",
    token=HF_TOKEN,
    commit_message="Upload LoRA adapter + tokenizer",
)

print("Uploaded:", f"https://huggingface.co/{ADAPTER_REPO}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded: https://huggingface.co/huyle1901/qwen25-3b-legal-qlora
